In [ ]:
"""
is_bottleneck_slot Train 전용 재계산 (leakage 제거)

bottleneck_analysis.ipynb의 최종 로직(z<=-1.0 AND relative_drop>=0.20,
30분 단위)을 그대로 쓰되, 입력 데이터를 Train 기간(timestamp < 2026-06-14,
xgb_feature_matrix에서 쓰는 split 경계와 동일)으로만 제한해서 다시 계산한다.
임계값(Z_THRESH, DROP_THRESH)은 이미 민감도 검증을 마쳤으므로 그대로 유지 -
바뀌는 건 "어떤 기간의 데이터로 계산하느냐"뿐이다.

출력: output/eda/is_bottleneck_slot_TRAIN_ONLY.csv
  segment_key, time_slot, is_bottleneck_slot (+ 참고용 avg_speed, z, relative_drop)
"""

from pathlib import Path

import polars as pl

SPEED_PATH = "./output/features/speed_features.parquet"
OUTPUT_DIR = Path("./output/eda")
OUTPUT_PATH = OUTPUT_DIR / "is_bottleneck_slot_TRAIN_ONLY.csv"

TRAIN_END = "2026-06-14"  # xgb_feature_matrix_ver2/3와 동일한 split 경계
Z_THRESH = -1.0
DROP_THRESH = 0.20

print(f"Train 컷오프: timestamp < {TRAIN_END}")

In [ ]:
# ==================================================================
# 1. Train 기간만 필터링 + 시간 파생
# ==================================================================

df = pl.read_parquet(SPEED_PATH).filter(pl.col("timestamp") < pl.lit(TRAIN_END).str.to_datetime())

df = df.with_columns(
    [
        pl.col("timestamp").dt.hour().alias("hour"),
        (pl.col("timestamp").dt.weekday() - 1).alias("dow"),
        ((pl.col("timestamp").dt.minute() // 30) * 30).alias("minute_bin"),
    ]
).with_columns(
    [
        (pl.col("dow") >= 5).alias("is_weekend"),
        ((pl.col("hour").cast(pl.Int32) * 100) + pl.col("minute_bin")).alias("time_slot"),
    ]
)

print(f"Train 기간 행 수: {df.height}")
print(f"기간: {df['timestamp'].min()} ~ {df['timestamp'].max()}")

In [ ]:
# ==================================================================
# 2. 평일 기준 segment_key x time_slot 통계 + 병목 판정
# ==================================================================

hourly = (
    df.filter(~pl.col("is_weekend"))
    .group_by(["segment_key", "time_slot"])
    .agg(
        [
            pl.col("V_segment").mean().alias("avg_speed"),
            (pl.col("V_segment") < 20.0).mean().alias("pct_below_20"),
            (pl.col("V_segment") < 15.0).mean().alias("pct_below_15"),
        ]
    )
    .sort(["segment_key", "time_slot"])
)

stats = hourly.group_by("segment_key").agg(
    [
        pl.col("avg_speed").mean().alias("mean_slot"),
        pl.col("avg_speed").std().alias("std_slot"),
        pl.col("avg_speed").max().alias("free_flow"),
    ]
)

hourly = hourly.join(stats, on="segment_key").with_columns(
    [
        ((pl.col("avg_speed") - pl.col("mean_slot")) / pl.col("std_slot")).alias("z"),
        ((pl.col("free_flow") - pl.col("avg_speed")) / pl.col("free_flow")).alias("relative_drop"),
    ]
).with_columns(
    ((pl.col("z") <= Z_THRESH) & (pl.col("relative_drop") >= DROP_THRESH)).alias("is_bottleneck_slot")
).with_columns(
    (pl.col("relative_drop") * pl.col("pct_below_20")).alias("severity_score")
)

hourly.write_csv(OUTPUT_PATH)

n_segments = hourly["segment_key"].n_unique()
n_no_rh = (
    hourly.group_by("segment_key")
    .agg(pl.col("is_bottleneck_slot").sum().alias("n"))
    .filter(pl.col("n") == 0)
    .height
)
print(f"segment_key 수: {n_segments}, rush hour 없는 구간: {n_no_rh}")
print(f"저장 완료: {OUTPUT_PATH.resolve()}")

In [ ]:
# ==================================================================
# 3. 전체기간 버전과 비교 (얼마나 달라졌는지 확인)
# ==================================================================

full_period = pl.read_csv("./output/eda/is_bottleneck_slot_FINAL_named.csv").select(
    ["segment_key", "time_slot", pl.col("is_bottleneck_slot").alias("is_bottleneck_slot_full")]
)
train_only = hourly.select(["segment_key", "time_slot", "is_bottleneck_slot"])

compare = train_only.join(full_period, on=["segment_key", "time_slot"], how="inner")
n_diff = compare.filter(pl.col("is_bottleneck_slot") != pl.col("is_bottleneck_slot_full")).height
print(f"전체 슬롯 수: {compare.height}")
print(f"판정이 달라진 슬롯 수: {n_diff} ({n_diff / compare.height:.1%})")